In [17]:
import re
import os
import random
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.common.exceptions import NoSuchElementException, TimeoutException

import pandas as pd
import numpy as np
import streamlit as st

#1° ETAPA (DISPARAR CLIENTES NA LISTA);
#2° ETAPA (ATENDER CLIENTES NA FILA AUTOMATICAMENTE)
#3° ETAPA (CAPTURAR INDICADORES DOS ATENDIMENTOS DURANTE A AUTOMAÇÃO)


In [18]:
def get_driver(browser=None):

    if browser == "chrome":
        return webdriver.Chrome()

    elif browser == "firefox":
        return webdriver.Firefox()

    elif browser == "edge":
        return webdriver.Edge()

    elif browser == "brave":
        options = Options()
        options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
        return webdriver.Chrome(options=options)

    else:
        raise ValueError("Sem driver no selenium")


driver = get_driver("brave")  
driver.get("https://prd-sac.onrender.com/sistema/lista_atendimentos.php")

wait = WebDriverWait(driver, 0.5)

In [19]:
#Faz a captura dos atendimentos atuais na lista, i
def extrair_protocolos(driver):

    df_protocolos = []

    tr = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//table/tbody/tr')))
    trs = len(tr)

    for i in range(1, trs + 1):

        col = driver.find_elements(By.XPATH, f'//table/tbody/tr[{i}]/td')
        aberto = ""

        if len(col) >= 8:


            status_text = col[5].text.strip()
            print(f"{i}° ATENDIMENTO - PROTOCOLO: {col[0].text.strip()}" )
            print(status_text)

            try:
                botao = wait.until(EC.element_to_be_clickable((By.XPATH, f'/html/body/div/table/tbody/tr[{i}]/td[8]/a/button')))      
                aberto = botao.text.strip()
            except (NoSuchElementException, TimeoutException):
                pass

            status_aberto = "Em aberto" if "Assumir" in aberto else ("Em atendimento" if "Chat" in aberto else "Encerrado") 
            status_concluido = "Sim" if "CONCLUÍDO" in status_text else "Não"
            status_expirado = "Sim" if "EXPIRADO" in status_text else "Não"
            status_cancelado = "Sim" if "CANCELADO" in status_text else "Não"

            df_protocolos.append({
            "protocolo": col[0].text.strip(),
            "servico": col[2].text.strip(),
            "tipo_servico": col[3].text.strip(),
            "validade": col[4].text.strip(),
            "feedback": col[6].text.strip(),
            "abertos": status_aberto,
            "concluidos": status_concluido,
            "expirados": status_expirado,
            "cancelados": status_cancelado
            })

            print("=" * 50)
            print(f"PROTOCOLO:  {df_protocolos[-1]}")
            print("="*50)

             
    df = pd.DataFrame(df_protocolos)
    return df

extrair_protocolos(driver)

1° ATENDIMENTO - PROTOCOLO: BIOME.004.124-0
ABERTO
PROTOCOLO:  {'protocolo': 'BIOME.004.124-0', 'servico': 'BIOMETRIA_DIGITAL', 'tipo_servico': 'Não especificado', 'validade': '21/09/2026 14:38:45', 'feedback': '-', 'abertos': 'Em aberto', 'concluidos': 'Não', 'expirados': 'Não', 'cancelados': 'Não'}
2° ATENDIMENTO - PROTOCOLO: BIOME.003.123-0
ABERTO
PROTOCOLO:  {'protocolo': 'BIOME.003.123-0', 'servico': 'BIOMETRIA_DIGITAL', 'tipo_servico': 'Não especificado', 'validade': '21/09/2026 14:38:41', 'feedback': '-', 'abertos': 'Em aberto', 'concluidos': 'Não', 'expirados': 'Não', 'cancelados': 'Não'}
3° ATENDIMENTO - PROTOCOLO: BIOME.002.122-0
ABERTO
PROTOCOLO:  {'protocolo': 'BIOME.002.122-0', 'servico': 'BIOMETRIA_DIGITAL', 'tipo_servico': 'Não especificado', 'validade': '21/09/2026 14:38:36', 'feedback': '-', 'abertos': 'Em aberto', 'concluidos': 'Não', 'expirados': 'Não', 'cancelados': 'Não'}
4° ATENDIMENTO - PROTOCOLO: BIOME.001.121-0
ABERTO
PROTOCOLO:  {'protocolo': 'BIOME.001.121-0'

,protocolo,servico,tipo_servico,validade,feedback,abertos,concluidos,expirados,cancelados
0,BIOME.004.124-0,BIOMETRIA_DIGITAL,Não especificado,21/09/2026 14:38:45,-,Em aberto,Não,Não,Não
1,BIOME.003.123-0,BIOMETRIA_DIGITAL,Não especificado,21/09/2026 14:38:41,-,Em aberto,Não,Não,Não
2,BIOME.002.122-0,BIOMETRIA_DIGITAL,Não especificado,21/09/2026 14:38:36,-,Em aberto,Não,Não,Não
3,BIOME.001.121-0,BIOMETRIA_DIGITAL,Não especificado,21/09/2026 14:38:31,-,Em aberto,Não,Não,Não
4,VALID.013.120-0,VALIDAÇÃO,Validação de atendimento,21/09/2026 14:13:52,5,Encerrado,Sim,Não,Não
...,...,...,...,...,...,...,...,...,...
119,CONFE.000.005-0,CONFERENCIA,Videoconferência de E-CNPJ,20/09/2026 16:45:00,3,Encerrado,Sim,Não,Não
120,VALID.000.004-0,VALIDAÇÃO,Não especificado,14/09/2026 09:00:00,-,Encerrado,Sim,Não,Não
121,CONFE.000.003-0,CONFERENCIA,Não especificado,07/09/2026 11:15:00,-,Encerrado,Não,Não,Sim
122,BIOME.000.002-0,BIOMETRIA_DIGITAL,Não especificado,15/09/2026 10:30:00,-,Encerrado,Não,Sim,Não


In [20]:
# def relatorio_de_atendimentos(df):

#     df_relatorio = pd.DataFrame(df)
#     total_abertos = df_relatorio['abertos'].value_counts()['Em aberto']
#     total_concluído = df_relatorio['concluidos'].value_counts()['Sim']
#     total_expirados = df_relatorio['expirados'].value_counts()['Sim']
#     total_cancelados = df_relatorio['cancelados'].value_counts()['Sim']
#     media_feedback = df_relatorio['feedback'].mean()
#     servico_em_alta = df_relatorio['servico'].value_counts().max()

    


    
